#Crearemos order_analysis


In [0]:
%sql
CREATE OR REPLACE TEMP VIEW vw_orders_base AS
SELECT
    orders.*,
    customers.customer_unique_id,
    customers.zip_code_prefix,
    customers.state AS customer_state,
    customers.city AS customer_city
FROM olist.silver.orders AS orders
LEFT JOIN olist.silver.customers AS customers
    ON orders.customer_id = customers.customer_id;
 

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW vw_payments_by_orders AS
SELECT
    order_id,
    COUNT(*) AS cantidad_pagos,
    SUM(payment_value) AS total_paid,
    MAX(payment_sequential) AS max_sequential,
    MAX(payment_installments) AS max_installments,
    COLLECT_SET(payment_type) AS payment_types
FROM olist.silver.order_payments
GROUP BY order_id;


In [0]:
%sql
SELECT order_id, COUNT(*)
FROM vw_payments_by_orders
GROUP BY order_id
HAVING COUNT(*) > 1;

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW vw_items_by_orders AS
SELECT
    order_id,
    COUNT(*) AS items_count,
    SUM(price) AS total_price,
    SUM(freight_value) AS total_freight_value,
    COUNT(DISTINCT product_id) AS distinct_products,
    COUNT(DISTINCT seller_id) AS distinct_sellers
FROM olist.silver.order_items
GROUP BY order_id;
    

In [0]:
%sql
SELECT order_id, COUNT(*)
FROM vw_items_by_orders
GROUP BY order_id
HAVING COUNT(*) > 1;

In [0]:
%sql
select order_id, count(*) from olist.silver.order_reviews group by order_id having count(*) > 1


In [0]:
%sql
CREATE OR REPLACE TEMP VIEW vw_reviews_by_orders AS
SELECT
    order_id,
    COUNT(*) AS reviews_count,
    AVG(review_score) AS avg_review_score,
    MIN(review_score) AS min_review_score,
    MAX(review_score) AS max_review_score,
    MAX(
        CASE 
            WHEN review_score <= 2 THEN 1 
            ELSE 0 
        END
    ) AS has_negative_review,
    MAX(review_creation_date) AS last_review_date
FROM olist.silver.order_reviews
GROUP BY order_id;
    

In [0]:
%sql
SELECT order_id, COUNT(*)
FROM vw_reviews_by_orders
GROUP BY order_id
HAVING COUNT(*) > 1;

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW vw_order_analysis AS
SELECT
    orders.*,

    payments.cantidad_pagos,
    payments.total_paid,
    payments.max_sequential,
    payments.max_installments,
    payments.payment_types,

    items.items_count,
    items.total_price,
    items.total_freight_value,
    items.distinct_products,
    items.distinct_sellers,

    reviews.reviews_count,
    reviews.avg_review_score,
    reviews.min_review_score,
    reviews.max_review_score,
    reviews.has_negative_review,
    reviews.last_review_date

FROM vw_orders_base AS orders

LEFT JOIN vw_payments_by_orders AS payments
    ON orders.order_id = payments.order_id

LEFT JOIN vw_items_by_orders AS items
    ON orders.order_id = items.order_id

LEFT JOIN vw_reviews_by_orders AS reviews
    ON orders.order_id = reviews.order_id;


In [0]:
%sql
SELECT 'gold' as tabla, COUNT(*) as cantidad FROM vw_order_analysis
union ALL
select 'silver' as tabla,count(*) as cantidad FROM olist.silver.orders

In [0]:
%sql
SELECT order_id, COUNT(*)
FROM vw_order_analysis
GROUP BY order_id
HAVING COUNT(*) > 1;

In [0]:
%sql
CREATE OR REPLACE TABLE olist.gold.order_analysis AS
SELECT * FROM vw_order_analysis;

# Resumen - Gold Order Analysis

En este notebook se inició la construcción de la capa Gold del proyecto, utilizando las tablas previamente preparadas en la capa Silver.

El objetivo principal fue crear una vista analítica con granularidad de **un registro por pedido**, integrando información de clientes, pagos, productos y evaluaciones.

## View Orders Base

Se creó la vista `vw_orders_base`.

### Transformaciones realizadas

- Se utilizó `olist.silver.orders` como tabla principal.
- Se realizó un `LEFT JOIN` con `olist.silver.customers`.
- Se incorporó información geográfica del cliente:
  - código postal;
  - estado;
  - ciudad.
- Se conservó `customer_unique_id`.
- Se mantuvieron todos los pedidos aunque no existiera información asociada del cliente.

---

## View Payments by Orders

Se creó la vista `vw_payments_by_orders`.

### Agregaciones realizadas

Los pagos fueron agrupados por `order_id` para mantener una granularidad de un registro por pedido.

Se calcularon:

- cantidad de registros de pago;
- valor total pagado;
- máximo valor de `payment_sequential`;
- máximo número de cuotas;
- conjunto de tipos de pago utilizados.

Esto permite representar correctamente pedidos que contienen más de un método o registro de pago.

---

## View Items by Orders

Se creó la vista `vw_items_by_orders`.

### Agregaciones realizadas

Los items fueron agrupados por `order_id`.

Se calcularon:

- cantidad total de items;
- valor total de los productos;
- valor total del flete;
- cantidad de productos distintos;
- cantidad de vendedores distintos.

De esta forma se evitó duplicar pedidos durante los joins posteriores.

---

## View Reviews by Orders

Antes de construir la vista se comprobó que algunos pedidos poseen más de una review.

Por este motivo, las evaluaciones fueron agrupadas por `order_id`.

### Métricas generadas

- cantidad de reviews;
- puntuación promedio;
- puntuación mínima;
- puntuación máxima;
- indicador de existencia de al menos una review negativa;
- fecha de la última review.

Se definió como review negativa aquella con `review_score <= 2`.

---

## View Order Analysis

Finalmente, se creó la vista:

`vw_order_analysis`

Esta vista integra:

- información del pedido;
- información del cliente;
- métricas de pagos;
- métricas de items;
- métricas de reviews.

Se utilizaron `LEFT JOIN` tomando los pedidos como universo principal del análisis.

## Validaciones finales

Se verificó que:

- el número de registros de `vw_order_analysis` sea igual al número de pedidos;
- `order_id` permanezca único;
- los joins no generen duplicación de pedidos.
- Luego de verificar estas validaciones se crea la tabla `order_analysis`

## Resultado

Se obtuvo una tabla analítica con granularidad:

**1 fila = 1 pedido**

Esta tabla funcionará como base para generar métricas de negocio relacionadas con:

- retrasos en entregas;
- experiencia del cliente;
- comportamiento de pagos;
- costos de productos y flete;
- participación de productos y vendedores.

#Enriqueser `Order_analysis`

In [0]:
%sql
SELECT
    order_id,
    order_purchase_timestamp,
    order_delivered_customer_date,
    order_estimated_delivery_date,

    DATEDIFF(
        order_delivered_customer_date,
        order_purchase_timestamp
    ) AS delivery_days,

    DATEDIFF(
        order_estimated_delivery_date,
        order_purchase_timestamp
    ) AS estimated_delivery_days,

    DATEDIFF(
        order_delivered_customer_date,
        order_estimated_delivery_date
    ) AS delay_days,

    CASE
        WHEN total_price > 0
        THEN (total_freight_value / total_price) * 100
        ELSE NULL
    END AS freight_percentage

FROM olist.gold.order_analysis;

In [0]:
%sql
CREATE OR REPLACE TEMP view vw_analysis AS
SELECT 
    order_id,
    DATEDIFF(order_delivered_customer_date, order_purchase_timestamp) AS delivery_days,
    DATEDIFF(order_estimated_delivery_date, order_purchase_timestamp) AS estimated_delivery_days,
    DATEDIFF(order_delivered_customer_date, order_estimated_delivery_date) AS delay_days,
    CASE
        WHEN total_price > 0
            THEN (total_freight_value / total_price) *100
            ELSE NULL
    END AS freight_percentage
FROM olist.gold.order_analysis

In [0]:
%sql
Select * from vw_analysis

In [0]:
%sql 
describe vw_analysis

In [0]:
%sql
alter table olist.gold.order_analysis
add columns(
    delivery_days INT,
    estimated_delivery_days INT,
    delay_days INT,
    freight_percentage DOUBLE);


merge into olist.gold.order_analysis as main
using vw_analysis as develop
on main.order_id = develop.order_id
when matched then 
    update set 
        main.delivery_days = develop.delivery_days,
        main.estimated_delivery_days = develop.estimated_delivery_days,
        main.delay_days = develop.delay_days,
        main.freight_percentage = develop.freight_percentage




In [0]:
%sql
select count(*) from olist.gold.order_analysis

In [0]:
%sql
SELECT 
  COUNT(*) FILTER (WHERE delivery_days IS NULL) AS null_delivery_days,
  COUNT(*) FILTER (WHERE delay_days IS NULL) AS null_delay_days
FROM olist.gold.order_analysis


In [0]:
%sql
SELECT 
    COUNT(*) AS total,
    SUM(CASE WHEN (delay_days > 0 AND is_late_delivery = true) OR (delay_days <= 0 AND (is_late_delivery = false OR is_late_delivery IS NULL)) THEN 1 ELSE 0 END) AS matches,
    SUM(CASE WHEN (delay_days > 0 AND (is_late_delivery = false OR is_late_delivery IS NULL)) OR (delay_days <= 0 AND is_late_delivery = true) THEN 1 ELSE 0 END) AS mismatches
FROM olist.gold.order_analysis;

In [0]:
%sql
SELECT
    COUNT(*) AS same_day_but_timestamp_late
FROM olist.gold.order_analysis
WHERE delay_days = 0
  AND is_late_delivery = TRUE;

In [0]:
%sql
ALTER TABLE olist.gold.order_analysis
ADD COLUMNS (
    is_late_delivery_business BOOLEAN
);

In [0]:
%sql
UPDATE olist.gold.order_analysis
SET is_late_delivery_business =
    CASE
        WHEN delay_days > 0 THEN TRUE
        WHEN delay_days <= 0 THEN FALSE
        ELSE NULL
    END;


## Enriquecimiento de `gold.order_analysis`

Después de construir la tabla principal `olist.gold.order_analysis`, se añadieron nuevas variables derivadas para facilitar el análisis del comportamiento de las entregas.

### Variables calculadas

Se creó inicialmente una vista auxiliar para comprobar el comportamiento de las métricas antes de incorporarlas a la tabla Gold.

Las variables calculadas fueron:

- `delivery_days`: número de días transcurridos entre la fecha de compra y la fecha real de entrega.
- `estimated_delivery_days`: número de días previstos entre la compra y la fecha estimada de entrega.
- `delay_days`: diferencia en días entre la fecha real de entrega y la fecha estimada.
  - Valores positivos representan entregas tardías.
  - Valor `0` representa entrega en la fecha prevista.
  - Valores negativos representan entregas anticipadas.
- `freight_percentage`: porcentaje que representa el coste del flete respecto al valor total de los productos del pedido.

### Validación previa

Antes de modificar la tabla Gold se utilizó una vista auxiliar para inspeccionar los resultados de estas métricas.

Posteriormente se agregó `order_id` a la vista para mantener una clave única que permitiera relacionarla con `order_analysis`.

### Enriquecimiento mediante Delta Lake

Las nuevas columnas fueron añadidas a `olist.gold.order_analysis` mediante `ALTER TABLE`.

Después se utilizó `MERGE INTO` para actualizar los pedidos existentes con las métricas calculadas en la vista auxiliar.

El `MERGE` actualizó **99.441 registros**, sin insertar ni eliminar filas.

### Validación de valores nulos

Se encontraron:

- `2.965` registros con `delivery_days` nulo.
- `2.965` registros con `delay_days` nulo.

Estos registros se conservaron, ya que corresponden a pedidos donde no existe una fecha real de entrega disponible.

### Revisión del indicador de retraso

Se comparó `delay_days` con el indicador original `is_late_delivery`.

Se detectaron diferencias debido a que:

- `is_late_delivery` compara los `TIMESTAMP` completos.
- `delay_days` utiliza una diferencia a nivel de días.

Por ejemplo, un pedido entregado durante el mismo día estimado, pero algunas horas después del timestamp estimado, podía ser clasificado como retrasado por el indicador original aunque `delay_days = 0`.

### Indicador de retraso para análisis de negocio

Para mantener ambas interpretaciones se conservó el indicador original y se creó:

`is_late_delivery_business`

La nueva variable considera:

- `TRUE` cuando `delay_days > 0`.
- `FALSE` cuando `delay_days <= 0`.
- `NULL` cuando no existe información suficiente para calcular el retraso.

Este indicador será utilizado como referencia principal para las métricas posteriores relacionadas con retrasos y experiencia del cliente.

## Resultado

`olist.gold.order_analysis` quedó enriquecida con métricas temporales y comerciales que permiten comenzar el análisis de negocio sobre entregas, retrasos, costes y satisfacción del cliente.

#Delivery metrics

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW vw_delivery_metrics AS
SELECT
    COUNT(*) AS total_orders,

    COUNT(*) FILTER (
        WHERE order_status = 'DELIVERED'
    ) AS delivered_orders,

    COUNT(*) FILTER (
        WHERE is_late_delivery_business = TRUE
    ) AS late_deliveries,

    ROUND(
        COUNT(*) FILTER (WHERE is_late_delivery_business = TRUE) * 100.0
        / COUNT(*) FILTER (WHERE is_late_delivery_business IS NOT NULL),
        2
    ) AS late_delivery_percentage,

    ROUND(AVG(delivery_days), 2) AS avg_delivery_days,

    ROUND(
        AVG(delay_days) FILTER (WHERE delay_days > 0),
        2
    ) AS avg_delay_days,

    MAX(delay_days) AS max_delay_days

FROM olist.gold.order_analysis;

In [0]:
%sql
Select * from vw_delivery_metrics

In [0]:
%sql
SELECT 
order_id, order_status, order_delivered_customer_date
FROM olist.gold.order_analysis
WHERE order_status = 'DELIVERED'
  AND order_delivered_customer_date IS NULL;

In [0]:
%sql
SELECT
    order_status,
    COUNT(*) AS cantidad
FROM olist.gold.order_analysis
WHERE order_delivered_customer_date IS NOT NULL
  AND order_status <> 'DELIVERED'
GROUP BY order_status;

In [0]:
%sql
CREATE OR REPLACE TABLE olist.gold.delivery_metrics AS
SELECT *
FROM vw_delivery_metrics;

In [0]:
%sql
select * from olist.gold.delivery_metrics

In [0]:
%sql
CREATE OR REPLACE VIEW vw_delivery_review_metrics AS
SELECT
    is_late_delivery_business,
    COUNT(*) AS total_orders,
    COUNT(avg_review_score) AS orders_with_review,
    ROUND(AVG(avg_review_score), 2) AS avg_review_score,

    COUNT(*) FILTER (
        WHERE has_negative_review = 1
    ) AS negative_reviews,

    ROUND(
        COUNT(*) FILTER (WHERE has_negative_review = 1) * 100.0
        / COUNT(avg_review_score),
        2
    ) AS negative_review_percentage

FROM olist.gold.order_analysis
WHERE is_late_delivery_business IS NOT NULL
GROUP BY is_late_delivery_business;

In [0]:
%sql
select * from vw_delivery_review_metrics;

In [0]:
%sql
create or replace table olist.gold.customer_satisfaction_metrics AS 
select * from vw_delivery_review_metrics

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW classified_orders AS
    SELECT
        order_id,
        delay_days,
        avg_review_score,
        has_negative_review,

        CASE
            when delay_days < 0 then 'EARLY'
            when delay_days =0 then 'ON_TIME'
            when delay_days between 1 and 3 then 'LATE_1_3'
            when delay_days between 4 and 7 then 'LATE_4_7'
            when delay_days between 8 and 15 then 'LATE_8_15'
            when delay_days > 15 then 'LATE_16_PLUS'
        END AS delay_severity

    FROM olist.gold.order_analysis

    WHERE delay_days IS NOT NULL


In [0]:
%sql
Select * from classified_orders

In [0]:
%sql
SELECT
    delay_severity,
    COUNT(*) AS total_orders,
    COUNT(avg_review_score) AS orders_with_review,
    ROUND(AVG(avg_review_score), 2) as avg_review_score,

    COUNT(*) FILTER (
        WHERE has_negative_review = 1
    ) AS negative_reviews,
    round(COUNT(*) FILTER (WHERE has_negative_review = 1) * 100.0,2) as negative_review_percentage
FROM classified_orders
GROUP BY delay_severity

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW vw_delay_severity_metrics AS

WITH classified_orders AS (

    SELECT
        order_id,
        delay_days,
        avg_review_score,
        has_negative_review,

        CASE
            when delay_days < 0 then 'EARLY'
            when delay_days =0 then 'ON_TIME'
            when delay_days between 1 and 3 then 'LATE_1_3'
            when delay_days between 4 and 7 then 'LATE_4_7'
            when delay_days between 8 and 15 then 'LATE_8_15'
            when delay_days > 15 then 'LATE_16_PLUS'
        END AS delay_severity

    FROM olist.gold.order_analysis
    WHERE delay_days IS NOT NULL
)

SELECT
    delay_severity,
    COUNT(*) AS total_orders,
    COUNT(avg_review_score) AS orders_with_review,
    ROUND(COUNT(*) FILTER (WHERE has_negative_review = 1) * 100.0/ COUNT(avg_review_score),2) AS negative_review_percentage,
    COUNT(*) FILTER (
        WHERE has_negative_review = 1
    ) AS negative_reviews,
    ROUND(AVG(avg_review_score), 2) AS avg_review_score
FROM classified_orders
GROUP BY delay_severity

In [0]:
%sql
select * from vw_delay_severity_metrics

In [0]:
%sql
Select count(*) from olist.gold.order_analysis where delay_days is not null
union all
select sum(total_orders) from vw_delay_severity_metrics

In [0]:
%sql
SELECT
    delay_severity,
    avg_review_score,
    negative_review_percentage
FROM vw_delay_severity_metrics
ORDER BY
    CASE delay_severity
        WHEN 'EARLY' THEN 1
        WHEN 'ON_TIME' THEN 2
        WHEN 'LATE_1_3' THEN 3
        WHEN 'LATE_4_7' THEN 4
        WHEN 'LATE_8_15' THEN 5
        WHEN 'LATE_16_PLUS' THEN 6
    END;

In [0]:
%sql
CREATE or replace table olist.gold.delay_severity_metrics as 
SELECT * from vw_delay_severity_metrics

In [0]:
%sql
Select * from olist.gold.delay_severity_metrics


# Resumen - Gold Business Metrics

En este notebook se construyó la capa Gold principal del proyecto a partir de las tablas previamente preparadas en Silver.

El objetivo fue integrar las diferentes entidades del dataset a nivel de pedido y generar métricas orientadas a analizar los retrasos en las entregas y su relación con la satisfacción del cliente.

---

## 1. Construcción de `order_analysis`

Se definió como granularidad principal:

**1 fila = 1 pedido (`order_id`)**

Para evitar duplicaciones durante los joins, las tablas con relaciones uno-a-muchos fueron agregadas previamente a nivel de pedido.

### Orders y Customers

Se creó una vista base utilizando `silver.orders` como tabla principal y realizando un `LEFT JOIN` con `silver.customers`.

Se incorporó información como:

- identificador único del cliente;
- código postal;
- ciudad;
- estado.

El pedido se mantuvo como entidad principal del análisis.

### Payments

Los pagos fueron agrupados por `order_id`.

Se calcularon métricas como:

- cantidad de registros de pago;
- valor total pagado;
- máximo `payment_sequential`;
- máximo número de cuotas;
- tipos de pago utilizados.

### Order Items

Los items fueron agrupados por `order_id`.

Se obtuvieron:

- cantidad de items;
- valor total de productos;
- valor total de flete;
- productos asociados;
- vendedores asociados;
- cantidad de productos distintos;
- cantidad de vendedores distintos.

### Reviews

Se comprobó previamente que algunos pedidos tienen más de una review.

Por este motivo las reviews fueron agrupadas por `order_id`, obteniendo:

- cantidad de reviews;
- puntuación promedio;
- puntuación mínima;
- puntuación máxima;
- existencia de al menos una review negativa;
- fecha de la última review.

Se consideró como review negativa una puntuación `<= 2`.

---

## 2. Tabla Gold principal

Después de validar las vistas intermedias se materializó:

`olist.gold.order_analysis`

La tabla mantiene una granularidad de un registro por pedido e integra información de:

- pedidos;
- clientes;
- pagos;
- items;
- reviews.

Se verificó que:

- el número de registros coincidiera con la cantidad original de pedidos;
- `order_id` permaneciera único después de los joins.

---

## 3. Enriquecimiento de `order_analysis`

Se añadieron nuevas variables analíticas:

- `delivery_days`: días transcurridos entre la compra y la entrega real.
- `estimated_delivery_days`: días previstos entre la compra y la fecha estimada de entrega.
- `delay_days`: diferencia entre la fecha real y la fecha estimada de entrega.
- `freight_percentage`: porcentaje que representa el flete respecto al valor de los productos.

La interpretación de `delay_days` es:

- valor negativo: pedido entregado antes de la fecha estimada;
- `0`: pedido entregado el día estimado;
- valor positivo: pedido entregado con retraso.

Las métricas fueron validadas inicialmente mediante una vista auxiliar y posteriormente incorporadas a `order_analysis` utilizando funcionalidades de Delta Lake.

Se utilizó `ALTER TABLE` para añadir las nuevas columnas y `MERGE INTO` para actualizar los registros existentes.

El `MERGE` actualizó los **99.441 pedidos**, sin insertar ni eliminar registros.

---

## 4. Validación de fechas de entrega

Se encontraron **2.965 pedidos** donde no fue posible calcular `delivery_days` ni `delay_days` debido a la ausencia de una fecha real de entrega.

También se detectaron inconsistencias entre `order_status` y las fechas registradas:

- **8 pedidos** con estado `DELIVERED` no tienen `order_delivered_customer_date`.
- **6 pedidos** tienen fecha real de entrega aunque su estado sea diferente de `DELIVERED`.

Estos registros fueron conservados para mantener la trazabilidad del dataset.

---

## 5. Definición de retraso para análisis de negocio

Se detectaron diferencias entre el indicador original `is_late_delivery` y `delay_days`.

El indicador original compara timestamps completos, mientras que `DATEDIFF` trabaja a nivel de días.

Por esta razón se creó:

`is_late_delivery_business`

La nueva definición utilizada para el análisis es:

- `TRUE`: `delay_days > 0`
- `FALSE`: `delay_days <= 0`
- `NULL`: no existe información suficiente para calcular el retraso.

De esta forma, un pedido entregado durante el mismo día estimado se considera entregado a tiempo independientemente de la hora.

---

# Métricas de negocio

## 6. Delivery Metrics

Se creó y posteriormente materializó:

`olist.gold.delivery_metrics`

Los principales resultados fueron:

- Total de pedidos: **99.441**
- Pedidos con estado `DELIVERED`: **96.478**
- Pedidos retrasados: **6.535**
- Porcentaje de pedidos retrasados: **6,77 %**
- Tiempo promedio de entrega: **12,5 días**
- Retraso promedio entre pedidos tardíos: **10,62 días**
- Retraso máximo observado: **188 días**

---

## 7. Retrasos y satisfacción del cliente

Se analizó la relación entre retrasos y evaluaciones de los clientes.

Posteriormente se materializó:

`olist.gold.customer_satisfaction_metrics`

### Pedidos sin retraso

- Pedidos: **89.941**
- Pedidos con review: **89.448**
- Review promedio: **4,29**
- Reviews negativas: **8.328**
- Porcentaje de reviews negativas: **9,31 %**

### Pedidos retrasados

- Pedidos: **6.535**
- Pedidos con review: **6.382**
- Review promedio: **2,27**
- Reviews negativas: **3.986**
- Porcentaje de reviews negativas: **62,46 %**

### Hallazgo

Los pedidos retrasados presentan una satisfacción considerablemente menor.

El porcentaje de reviews negativas aumenta de **9,31 %** en pedidos sin retraso a **62,46 %** en pedidos retrasados.

La review promedio también disminuye de **4,29** a **2,27**.

Estos resultados muestran una fuerte asociación entre los retrasos en las entregas y una peor experiencia del cliente.

---

## 8. Severidad del retraso

Para analizar si la satisfacción cambia conforme aumenta el retraso, los pedidos fueron clasificados en:

- `EARLY`: entrega anticipada.
- `ON_TIME`: entrega el día previsto.
- `LATE_1_3`: retraso entre 1 y 3 días.
- `LATE_4_7`: retraso entre 4 y 7 días.
- `LATE_8_15`: retraso entre 8 y 15 días.
- `LATE_16_PLUS`: retraso superior a 15 días.

Se creó y materializó:

`olist.gold.delay_severity_metrics`

### Resultados

| Severidad | Review promedio | Reviews negativas |
|---|---:|---:|
| EARLY | 4,29 | 9,26 % |
| ON_TIME | 4,04 | 12,50 % |
| LATE_1_3 | 3,29 | 32,18 % |
| LATE_4_7 | 2,11 | 67,68 % |
| LATE_8_15 | 1,68 | 80,02 % |
| LATE_16_PLUS | 1,73 | 78,39 % |

La suma de pedidos de todas las categorías fue validada contra los registros con `delay_days` disponible, obteniendo **96.476 pedidos en ambos casos**.

### Hallazgo

Los resultados muestran una clara tendencia de deterioro de la satisfacción conforme aumenta la severidad del retraso.

Los pedidos anticipados presentan una review promedio de **4,29**, mientras que los pedidos con retrasos superiores a varios días presentan evaluaciones cercanas a **2 o inferiores**.

El porcentaje de reviews negativas aumenta especialmente a partir de retrasos superiores a 4 días.

La relación no es perfectamente lineal entre todas las categorías, pero la tendencia general muestra una fuerte asociación entre mayor retraso y peor satisfacción.

---

# Resultado final

Al finalizar este notebook se construyeron las principales tablas analíticas de la capa Gold:

- `olist.gold.order_analysis`
- `olist.gold.delivery_metrics`
- `olist.gold.customer_satisfaction_metrics`
- `olist.gold.delay_severity_metrics`

La capa Gold permite analizar el ciclo completo del pedido desde una perspectiva de negocio y proporciona evidencia para responder la problemática principal del proyecto:

**Los retrasos en las entregas están fuertemente asociados con evaluaciones más bajas y una mayor proporción de experiencias negativas de los clientes.**